## SISTEM DETEKSI FEEDBACK ESSAY MENGGUNAKAN BiLSTM
Program ini dibuat untuk menganalisis teks essay secara otomatis dan menghasilkan kategori penilaian, skor, serta feedback menggunakan model Deep Learning berbasis TensorFlow Functional API dengan arsitektur Bidirectional LSTM (BiLSTM).

## IMPORT LIBRARY

In [70]:
!pip install -q scikit-learn matplotlib seaborn

import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## CONFIG

In [71]:
MAX_VOCAB = 15000
MAX_LEN = 300

EMBED_DIM = 128

BATCH_SIZE = 32
EPOCHS = 20

SCORE_MIN = 1
SCORE_MAX = 6

DATASET_PATH = '/content/drive/MyDrive/SALC_Project/dataset/data_NLPAutoFeedback.csv'

os.makedirs("model", exist_ok=True)

MODEL_PATH = "model/salc_model.keras"
TOKENIZER_PATH = "model/tokenizer.pkl"
LABEL_ENC_PATH = "model/label_encoder.pkl"

##  LOAD DATASET

In [72]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(DATASET_PATH)

print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  essay_id                                          full_text  score  \
0  000d118  Many people have car where they live. The thin...      3   
1  000fe60  I am a scientist at NASA that is discussing th...      3   
2  001ab80  People always wish they had the same technolog...      4   
3  001bdc0  We all heard about Venus, the planet without a...      4   
4  002ba53  Dear, State Senator\n\nThis is a letter to arg...      3   

   word_count  char_count                                    clean_full_text  
0         498        2677  many people have car where they live the thing...  
1         332        1669  i am a scientist at nasa that is discussing th...  
2         550        3077  people always wish they had the same technolog...  
3         451        2701  we all heard about venus the planet without al...  
4         373        2208  dear state senat

## PREPROCESSING TEXT

In [73]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'\n+', ' ', text)

    text = re.sub(r'[^a-z0-9\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


def normalize_score(score):

    return (score - SCORE_MIN) / (SCORE_MAX - SCORE_MIN)


def score_to_kategori(score):

    if score >= 4.5:
        return "baik"

    elif score >= 3:
        return "cukup"

    else:
        return "kurang"


text_col = 'clean_full_text' if 'clean_full_text' in df.columns else 'full_text'

df['text_clean'] = df[text_col].fillna('').apply(clean_text)

df['kategori'] = df['score'].apply(score_to_kategori)

df['score_norm'] = df['score'].apply(normalize_score)

print(df[['score', 'kategori']].head())

   score kategori
0      3    cukup
1      3    cukup
2      4    cukup
3      4    cukup
4      3    cukup


## TOKENIZER DAN PADDING

In [74]:
tokenizer = Tokenizer(
    num_words=MAX_VOCAB,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(df['text_clean'])

with open(TOKENIZER_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)

seqs = tokenizer.texts_to_sequences(df['text_clean'])

X = pad_sequences(
    seqs,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print("Shape X:", X.shape)

Shape X: (17307, 300)


## LABEL ENCODER

In [75]:
le = LabelEncoder()

le.fit(['baik', 'cukup', 'kurang'])

with open(LABEL_ENC_PATH, 'wb') as f:
    pickle.dump(le, f)

y_kategori = le.transform(df['kategori'])

y_skor = df['score_norm'].values

print("Classes:", le.classes_)

Classes: ['baik' 'cukup' 'kurang']


## SPLIT DATA

In [76]:
X_train, X_val, yc_train, yc_val, ys_train, ys_val = train_test_split(
    X,
    y_kategori,
    y_skor,
    test_size=0.15,
    random_state=42,
    stratify=y_kategori
)

print("Train:", len(X_train))
print("Val:", len(X_val))


Train: 14710
Val: 2597


## MEMBANGUN MODEL DEEP LEARNING

## CLASS WEIGHT, WEIGHTED LOSS

In [77]:
n_total = len(y_kategori)

n_classes = len(le.classes_)

counts = np.bincount(y_kategori)

weights = n_total / (n_classes * counts)

CLASS_WEIGHT_TENSOR = tf.constant(
    weights,
    dtype=tf.float32
)

print("Class Weights:")
for i, cls in enumerate(le.classes_):
    print(cls, ":", weights[i])

def weighted_crossentropy(y_true, y_pred):

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true,
        y_pred
    )

    y_true = tf.cast(
        tf.reshape(y_true, [-1]),
        tf.int32
    )

    weights = tf.gather(
        CLASS_WEIGHT_TENSOR,
        y_true
    )

    loss = loss * weights

    return tf.reduce_mean(loss)

Class Weights:
baik : 5.123445825932505
cukup : 0.5652557319223986
kurang : 0.9655230125523012


## BUILD MODEL

In [78]:
def build_model(n_classes):

    inputs = keras.Input(shape=(MAX_LEN,))

    x = layers.Embedding(
        input_dim=MAX_VOCAB,
        output_dim=EMBED_DIM,
        mask_zero=True
    )(inputs)

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True
        )
    )(x)

    x = layers.GlobalMaxPooling1D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    shared = layers.Dense(64, activation='relu')(x)

    output_kategori = layers.Dense(
        n_classes,
        activation='softmax',
        name='output_kategori'
    )(shared)

    output_skor = layers.Dense(
        1,
        activation='sigmoid',
        name='output_skor'
    )(shared)

    model = Model(
        inputs=inputs,
        outputs=[output_kategori, output_skor]
    )

    return model


model = build_model(len(le.classes_))

model.compile(

    optimizer=keras.optimizers.Adam(learning_rate=0.001),

    loss={
        'output_kategori': weighted_crossentropy,
        'output_skor': 'mse'
    },

    loss_weights={
        'output_kategori': 2.0,
        'output_skor': 0.3
    },

    metrics={
        'output_kategori': ['accuracy'],
        'output_skor': ['mae']
    }
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'global_max_pooling1d_3' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 300, 128)  │  1,920,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 300)       │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 300, 128)  │     98,816 │ embedding_3[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ bidirectional_3[… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     16,512 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_kategori     │ (None, 3)         │        195 │ dense_7[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_skor (Dense) │ (None, 1)         │         65 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,043,844 (7.80 MB)

 Trainable params: 2,043,844 (7.80 MB)

 Non-trainable params: 0 (0.00 B)

## CALLBACKS

In [79]:
callbacks = [

    EarlyStopping(
        monitor='val_output_kategori_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max'
    ),

    ModelCheckpoint(
        MODEL_PATH,
        monitor='val_output_kategori_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

## TRAINING

In [80]:
history = model.fit(

    X_train,

    {
        'output_kategori': yc_train,
        'output_skor': ys_train
    },

    validation_data=(
        X_val,
        {
            'output_kategori': yc_val,
            'output_skor': ys_val
        }
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    verbose=1
)

Epoch 1/20
459/460 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2.0146 - output_kategori_accuracy: 0.3994 - output_kategori_loss: 1.0007 - output_skor_loss: 0.0438 - output_skor_mae: 0.1693
Epoch 1: val_output_kategori_accuracy improved from None to 0.52830, saving model to model/salc_model.keras

Epoch 1: finished saving model to model/salc_model.keras
460/460 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - loss: 1.8221 - output_kategori_accuracy: 0.4637 - output_kategori_loss: 0.9055 - output_skor_loss: 0.0364 - output_skor_mae: 0.1525 - val_loss: 1.4933 - val_output_kategori_accuracy: 0.5283 - val_output_kategori_loss: 0.7550 - val_output_skor_loss: 0.0270 - val_output_skor_mae: 0.1284 - learning_rate: 0.0010
Epoch 2/20
459/460 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.2406 - output_kategori_accuracy: 0.6559 - output_kategori_loss: 0.6168 - output_skor_loss: 0.0236 - output_skor_mae: 0.1216
Epoch 2: val_output_kategori_accuracy improved from 0.52830 to 0.67886, saving model to model/salc_model.

## LOAD BEST MODEL

In [81]:
from tensorflow.keras.models import load_model

model = load_model(
    MODEL_PATH,
    custom_objects={
        'weighted_crossentropy': weighted_crossentropy
    }
)

print("Best model loaded!")

Best model loaded!


## EVALUATION

In [82]:
pred_cat_proba, pred_skor_norm = model.predict(X_val)

pred_cat_idx = np.argmax(pred_cat_proba, axis=1)

pred_kategori = le.inverse_transform(pred_cat_idx)

true_kategori = le.inverse_transform(yc_val)

print(classification_report(
    true_kategori,
    pred_kategori
))

82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
              precision    recall  f1-score   support

        baik       0.37      0.29      0.33       169
       cukup       0.76      0.78      0.77      1531
      kurang       0.71      0.71      0.71       897

    accuracy                           0.72      2597
   macro avg       0.62      0.59      0.60      2597
weighted avg       0.72      0.72      0.72      2597



## SAVE MODEL

In [83]:
print("Model saved:")
print(MODEL_PATH)
print(TOKENIZER_PATH)
print(LABEL_ENC_PATH)

Model saved:
model/salc_model.keras
model/tokenizer.pkl
model/label_encoder.pkl


## INFERENCE FUNCTION

In [84]:
def kategori_to_feedback(kategori):

    feedback_map = {

        'baik':
            'Excellent answer! Your arguments are strong and clearly explained.',

        'cukup':
            'Good effort! Add more detail and supporting evidence.',

        'kurang':
            'This answer needs improvement. Try to explain your ideas more clearly.'
    }

    return feedback_map[kategori]


def predict_feedback(text):

    cleaned = clean_text(text)

    seq = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post'
    )

    pred_cat_proba, pred_skor_norm = model.predict(
        padded,
        verbose=0
    )

    pred_cat_proba = pred_cat_proba[0]

    cat_idx = int(np.argmax(pred_cat_proba))

    confidence = float(np.max(pred_cat_proba))


    skor_raw = float(pred_skor_norm[0][0])

    skor_raw = (
        skor_raw *
        (SCORE_MAX - SCORE_MIN)
    ) + SCORE_MIN

    skor_raw = max(
        SCORE_MIN,
        min(SCORE_MAX, skor_raw)
    )

    skor_100 = int(round(
        (
            (skor_raw - SCORE_MIN)
            /
            (SCORE_MAX - SCORE_MIN)
        ) * 100
    ))


    if skor_raw >= 4.5:
        kategori = 'baik'

    elif skor_raw >= 3:
        kategori = 'cukup'

    else:
        kategori = 'kurang'

    return {

        'kategori': kategori,

        'skor': skor_100,

        'feedback': kategori_to_feedback(kategori),

        'confidence': round(confidence, 3)
    }

## TESTING

In [85]:
sample = """
Photosynthesis is the process by which plants use sunlight,
water and carbon dioxide to produce glucose and oxygen.
"""

result = predict_feedback(sample)

print(result)

{'kategori': 'kurang', 'skor': 22, 'feedback': 'This answer needs improvement. Try to explain your ideas more clearly.', 'confidence': 0.989}


In [86]:
sample = """
Work. School. Party. No matter what you have on your to do list you still have to get there, so why are car sales going down and how is it benificial to our society? Teens and young adults can often be heard discussing what brand of car they one day hope to buy and all the modifications they will make to 'trick it out'. Many people though end up being 20 or older before they actually get that car they always talked about, and despite parents claim of laziness there could be more to it then that. So what advantages could there be to aÂ  world with fewer cars on the road?

The first topic that is at the forfront of the minds of most people just starting out their new life is cost. The world revolves around money, thats just the way it is, so many people are finding themselves cutting out things that they don't really need. Now I know what your thinking but before you begin your argument that you really need that blue mercedes you saw in the magazine think about just how much that cost compared to your last paycheck, not so exiting anymore right? With public transportation becomingh more and more avalible many people are choosing to take the public bus rather then buy their own car. Another thought is all those taxes you pay. What you thought they went to police and teacher's wages and building nice little parks for the kiddies? Well acording the Environmantal Protection Agency 80% goes into making Highways and other transport, and why do they need that much money? Because we drive 100 tons over them every day. Many of the more polluted citys are even begining to put laws into place banning or restricting traffic in some areas and placing fines on those who violate them. Between the purchase price and matnence as well as gas and the money used to fund roads and highways theres a lot of dough pouring into your car drives.

Now heres the topic I know your all tired of hearing about but its still very important, the environment. Car emmisions are the second biggest source of polllution in the atmosphere, first place going to powerplants and third place being cows, yes cows. Is a world blanketed in smog and smoke where the very air that sustains you is poison the kind of place you want to live in or, for that matter, raise kids in? Its not the the planets health we're talking about here its the health of everything living on it including us. While we're on the topic of health one of the greatest health issues in America is obesity, now imagine if everyone left their cars at home and walked or biked instead, the whole country would be healthier then ever before. And you know how when you go into the country the air smells so nice and clean, well that could be the city too if we just cut back on emmisions, we could have nice fresh air as well. Now think of your children, or friends or even yourself, just outside playing football when a car comes by and almost hits you, dozens of people die every day in car crashes that wouldnt if there were less cars on the road. So think of the health of your familly, your friends, your pets, and yourself.

Finally we have the cultural shift, many Americans are already choosing to carpool take public transport or even just walk and less and less are buying cars. This trend is so noticibly prominent that car companys are trying to come up with new ideas and products to keep their customers coming. Citys are coming up with new systems of bike and car borrowing as well as a partnering with the telecommunications industry to create cities in which "Pedestrian, bicycle, private cars, commercial and public transportation traffic are woven into a connected network to save time, conserve resources, lower emissions and imporve safty". So for those of you that can't stand to be left behind by the times jump on the band wagon and you can do your part to bring about a new cleaner era in the world.

Cars, the parasprites of a modern age, keep multiplying and destroying more and more. The emmisions given of by these monstrosities will linger more many a milenia poisoning the air we breath and devastating the world out children will inherit. Unless we do something to stop it now this travesty will continue to ravange all life on the planet untill we are on the brink of extinction, so do us all a favor and help to push this new trend of a car free world.
"""

print(predict_feedback(sample))

{'kategori': 'baik', 'skor': 84, 'feedback': 'Excellent answer! Your arguments are strong and clearly explained.', 'confidence': 1.0}


In [87]:
sample = """
In "The Challenge of Exploring Venus," the author includes many benefits and dangers that come with exploring Venus. Venus is known for its uninhabitable atmosphere and environment, but scientists believe that long ago Venus was the planet most similar to Earth. In this article, the author believes that humans should go beyond the limits of dangers and supports his or her idea well with inspiring points that motivate anyone to look past the boundaries. The author lists the extreme dangers of Venus, and then answers why scientists are so set on exploring it.

The author first regards the dangers of Venus. The author explains the problem to finish with a promising solution. The author gives dreadful facts that prove the planet is extremely inhospitable for any human and even some space crafts. The author states, "These conditions are far more extreme than anything humans encounter on Earth; such an environment would crush even a submarine accustomed to diving to the deepest parts of our oceans and would liquefy many metals," The author does not say this to frighten anyone but to blunty state what the NASA scientists are up against.

Further into the article, the author answers the important question "why?" Why would anyone want to put themselves at these risks? Why is Venus worth all of the trouble? The answer is, "Astronomers are fascinated by Venus because it may well once have been the most Earth-like planet in our solar system." The author wants people to realize that Venus has the potential to be almost identical to Earth, but the author also includes that scientists cannot be certain without further exploration. The potential of Venus being another part of our world is certainly going to excite many astronaunts and explorers to go make it certain.

Venus has the potential to hold an exciting new world, but the dangers astronaunts and scientists must face are lethal. The author paints a new possibility in this article by describing the potential Venus could hold. This article clearly shows how the author informs about the dangers of Venus but then answers why the scientists are to eager to get to know this planets. With the way the author gives inspiring points about exploring this planet, anyone can look past the dangers and see a new world.
"""

print(predict_feedback(sample))

{'kategori': 'cukup', 'skor': 46, 'feedback': 'Good effort! Add more detail and supporting evidence.', 'confidence': 0.991}


In [88]:
import os

os.makedirs("salc_deploy/model", exist_ok=True)
os.makedirs("salc_deploy/api", exist_ok=True)
os.makedirs("salc_deploy/utils", exist_ok=True)

print("Folder deploy berhasil dibuat!")

Folder deploy berhasil dibuat!


In [89]:
import shutil

shutil.copy("model/salc_model.keras", "salc_deploy/model/")
shutil.copy("model/tokenizer.pkl", "salc_deploy/model/")
shutil.copy("model/label_encoder.pkl", "salc_deploy/model/")

print("Model berhasil dipindahkan!")

Model berhasil dipindahkan!
